In [1]:
import pandas as pd
import numpy as np

In [19]:
df = pd.read_csv("orders_(1).csv")

In [20]:
df = df[df["Product"] == "MIS"]
df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status
1,16/12/21 15:08,BUY,ASHOKLEY,MIS,1000/1000,125.70,COMPLETE
3,16/12/21 14:13,BUY,TATAMOTORS,MIS,250/250,490.55,COMPLETE
4,16/12/21 13:54,BUY,ASHOKLEY,MIS,0/1000,127.10,CANCELLED
5,16/12/21 13:21,SELL,TATAMOTORS,MIS,250/250,492.10,COMPLETE
6,16/12/21 12:51,BUY,TATAMOTORS,MIS,0/250,490.80,CANCELLED
7,16/12/21 12:39,SELL,ASHOKLEY,MIS,1000/1000,125.96,COMPLETE
8,16/12/21 12:29,BUY,ASHOKLEY,MIS,2000/2000,125.70,COMPLETE
9,16/12/21 11:22,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE
10,16/12/21 10:47,SELL,ASHOKLEY,MIS,0/2000,124.45,CANCELLED
11,16/12/21 10:46,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE


In [6]:
df["Qty"] = df["Qty."].str.split("/").str[0].astype(int)
df["Qty"]

1     1000
3      250
4        0
5      250
6        0
7     1000
8     2000
9     2000
10       0
11    2000
12    1000
13    1000
Name: Qty, dtype: int64

In [8]:
df["Turnover"] = df["Qty"] * df["Avg. price"]
df["Turnover"] 

1     125700.0
3     122637.5
4          0.0
5     123025.0
6          0.0
7     125960.0
8     251400.0
9     251900.0
10         0.0
11    251900.0
12    125600.0
13    125650.0
Name: Turnover, dtype: float64

In [10]:
BROKERAGE_RATE = 0.0003
STT_RATE = 0.00025
ETC_RATE = 0.0000297
SEBI_RATE = 10 / 10000000
STAMP_RATE = 0.00003
GST_RATE = 0.18

In [18]:
df["Brokerage"] = np.minimum(df["Turnover"] * BROKERAGE_RATE, 20)
df["STT"] = np.where(df["Type"] == "SELL", df["Turnover"] * STT_RATE, 0)
df["ETC"] = df["Turnover"] * ETC_RATE
df["SEBI"] = df["Turnover"] * SEBI_RATE
df["Stamp"] = np.where(df["Type"] == "BUY", df["Turnover"] * STAMP_RATE, 0)
df["GST"] = (df["Brokerage"] + df["ETC"] + df["SEBI"]) * GST_RATE
df["Total Charges"] = df[["Brokerage","STT","ETC","SEBI","Stamp","GST"]].sum(axis=1)
df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status,Qty,Turnover,Brokerage,STT,ETC,SEBI,Stamp,GST,Total Charges
1,16/12/21 15:08,BUY,ASHOKLEY,MIS,1000/1000,125.70,COMPLETE,1000,125700.0,20.0,0.00000,3.733290,0.125700,3.771000,4.294618,31.924608
3,16/12/21 14:13,BUY,TATAMOTORS,MIS,250/250,490.55,COMPLETE,250,122637.5,20.0,0.00000,3.642334,0.122637,3.679125,4.277695,31.721791
4,16/12/21 13:54,BUY,ASHOKLEY,MIS,0/1000,127.10,CANCELLED,0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
5,16/12/21 13:21,SELL,TATAMOTORS,MIS,250/250,492.10,COMPLETE,250,123025.0,20.0,30.75625,3.653843,0.123025,0.000000,4.279836,58.812954
6,16/12/21 12:51,BUY,TATAMOTORS,MIS,0/250,490.80,CANCELLED,0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
7,16/12/21 12:39,SELL,ASHOKLEY,MIS,1000/1000,125.96,COMPLETE,1000,125960.0,20.0,31.49000,3.741012,0.125960,0.000000,4.296055,59.653027
8,16/12/21 12:29,BUY,ASHOKLEY,MIS,2000/2000,125.70,COMPLETE,2000,251400.0,20.0,0.00000,7.466580,0.251400,7.542000,4.989236,40.249216
9,16/12/21 11:22,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE,2000,251900.0,20.0,62.97500,7.481430,0.251900,0.000000,4.991999,95.700329
10,16/12/21 10:47,SELL,ASHOKLEY,MIS,0/2000,124.45,CANCELLED,0,0.0,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
11,16/12/21 10:46,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE,2000,251900.0,20.0,62.97500,7.481430,0.251900,0.000000,4.991999,95.700329


In [13]:
summary = df.groupby(["Instrument","Type"]).agg(
    Qty=("Qty","sum"),
    Turnover=("Turnover","sum"),
    Avg_price=("Avg. price", lambda x: np.average(x, weights=df.loc[x.index,"Qty"])),
    Brokerage=("Brokerage","sum"),
    STT=("STT","sum"),
    ETC=("ETC","sum"),
    SEBI=("SEBI","sum"),
    Stamp=("Stamp","sum"),
    GST=("GST","sum"),
    Total_Charges=("Total Charges","sum")
).reset_index()
summary


,Instrument,Type,Qty,Turnover,Avg_price,Brokerage,STT,ETC,SEBI,Stamp,GST,Total_Charges
0,ASHOKLEY,BUY,5000,628350.0,125.670,80.0,0.00000,18.661995,0.628350,18.850500,17.872262,136.013107
1,ASHOKLEY,SELL,5000,629760.0,125.952,60.0,157.44000,18.703872,0.629760,0.000000,14.280054,251.053686
2,TATAMOTORS,BUY,250,122637.5,490.550,20.0,0.00000,3.642334,0.122637,3.679125,4.277695,31.721791
3,TATAMOTORS,SELL,250,123025.0,492.100,20.0,30.75625,3.653843,0.123025,0.000000,4.279836,58.812954


In [16]:
pnl = df.pivot_table(index="Instrument", columns="Type", values="Turnover", aggfunc="sum", fill_value=0)
pnl["Gross PnL"] = pnl.get("SELL",0) - pnl.get("BUY",0)

charges_stock = df.groupby("Instrument")["Total Charges"].sum()

final = pnl.join(charges_stock)
final["Net PnL"] = final["Gross PnL"] - final["Total Charges"]
final["% Charges"] = (final["Total Charges"] / final["Gross PnL"]) * 100
final

,BUY,SELL,Gross PnL,Total Charges,Net PnL,% Charges
Instrument,,,,,,
ASHOKLEY,628350.0,629760.0,1410.0,387.066793,1022.933207,27.451546
TATAMOTORS,122637.5,123025.0,387.5,90.534745,296.965255,23.363805


In [17]:
with pd.ExcelWriter("Kite_PnL_Output.xlsx") as writer:
    df.to_excel(writer, sheet_name="Trade_Level", index=False)
    summary.to_excel(writer, sheet_name="Stock_Type_Summary", index=False)
    final.reset_index().to_excel(writer, sheet_name="Overall_PnL", index=False)